# Run Sampling
Orchestration pipeline for Part 1: Preprocessing, PyMC compilation, sampling, posterior predictive simulation, and saving everything to disk.


In [ ]:
# Imports and Initial Setup
import time
import os
import pickle
import config
import data_utils
import model_pymc

total_start = time.time()


## Section 1: Initiating Survey Data Generation


In [ ]:
# Simulate DORA data reorganization and generate raw data
print("--- PART 1 - SECTION 1: INITIATING SURVEY DATA GENERATION ---")
sec_start = time.time()

sim_years = [2023, 2025, 2026]
df_raw, true_params = data_utils.simulate_dora_data_reorg(
    sim_years_reorg=sim_years,
    lineage_map=config.REORG_LINEAGE_MAP,
    sim_n_resp_per_team_year=5,
    hierarchy_map=config.SURVEY_HIERARCHY,
    response_options_sim=config.RESPONSE_OPTIONS
)
print(f"Elapsed time for SECTION 1: {time.time() - sec_start:.2f} seconds")


## Section 2: Parsing Structural Layers & Index Mappings


In [ ]:
# Load and preprocess the data, creating structured mappings
print("\n--- PART 1 - SECTION 2: PARSING STRUCTURAL LAYERS & INDEX MAPPINGS ---")
sec_start = time.time()
df_long, struct_maps = data_utils.load_and_preprocess_data(df_raw, config.SURVEY_HIERARCHY)
print(f"Elapsed time for SECTION 2: {time.time() - sec_start:.2f} seconds")


## Section 3: Executing MCMC Sampling & PPC


In [ ]:
# Build and run the PyMC model using the preprocessed data
print("\n--- PART 1 - SECTION 3: EXECUTING MCMC SAMPLING & PPC ---")
sec_start = time.time()
idata, model = model_pymc.build_and_run_model(df_long, struct_maps)
print(f"Elapsed time for SECTION 3: {time.time() - sec_start:.2f} seconds")


## Section 4: Serializing Output Files to Disk


In [ ]:
# Save inference trace, DataFrames, and structural maps to disk
print("\n--- PART 1 - SECTION 4: SERIALIZING OUTPUT FILES TO DISK ---")
sec_start = time.time()

# Save InferenceData object natively (completely preserves chains and coords)
idata.to_netcdf(os.path.join(config.MODEL_DIR, "dora_inference_data.nc"))

# Save preprocessing DataFrames
df_raw.to_pickle(os.path.join(config.MODEL_DIR, "df_raw.pkl"))
df_long.to_pickle(os.path.join(config.MODEL_DIR, "df_long.pkl"))

# Save structural maps
with open(os.path.join(config.MODEL_DIR, "struct_maps.pkl"), "wb") as f:
    pickle.dump(struct_maps, f)

print("  Sampling trace saved. Exiting process to clear VRAM pool.")
print(f"Elapsed time for SECTION 4: {time.time() - sec_start:.2f} seconds")

print(f"\n=======================================================")
print(f"Part 1 Sampling Completed in {time.time() - total_start:.1f} total seconds.")
print(f"=======================================================")
